In [ ]:
# imports
import re
from pathlib import Path

In [ ]:
# slow imports
from adaptive_polish.strategy import AdaptivePolishMillingStrategy
from adaptive_polish.config import AdaptivePolishMillingConfig

from fibsem.structures import FibsemImage

In [ ]:
base_path = (
    Path.home()
    / "OneDrive - The Rosalind Franklin Institute"
    / "Documents"
    / "test data"
    / "adaptive milling"
)

model_path = (
    base_path / "sem_models" / "Gen1" / "gen01_V8_FPN_RGB" / "cryo_sem_epoch_69.pth"
)
model_gen = "1.4fpn"

In [ ]:
config = AdaptivePolishMillingConfig(
    model_path=model_path,
    model_generation=model_gen,
    window_size_px=10,
    max_crack_area_um2=0.5,
    gis_stop_um=0.25,
    align_sem=False,
)

In [ ]:
strategy = AdaptivePolishMillingStrategy(config)
strategy._load_model()

In [ ]:
def run_on_directory(ap_directory: Path) -> None:
    image_regex = re.compile(r"^([\w_\-\. ]+)_img_(\d{3})_(?:SEM|FIB)$", re.I)

    sem_directory = ap_directory / "sem"
    fib_directory = ap_directory / "fib"
    plots_dir = ap_directory / "gis_plots"
    plots_dir.mkdir(exist_ok=True)
    for image_path in sem_directory.glob("*.tif"):
        match = image_regex.match(image_path.stem)
        if match is None:
            continue
        identifier = f"{match.group(1)}_img_{match.group(2)}"
        milling_cycle = int(match.group(2))
        fib_image_name = f"{identifier}_FIB.tif"
        try:
            strategy._check_lamella(
                milling_cycle=milling_cycle,
                image_name=identifier,
                fib_image=FibsemImage.load(str(fib_directory / fib_image_name)),
                sem_image=FibsemImage.load(str(image_path)),
                plots_folder=plots_dir,
                results_dict={},
            )
        except Exception:
            pass

In [ ]:
run_on_directory(
    base_path
    / "20250610_quick_ON_test"
    / "AutoLamella-2025-06-10-21-08"
    / "02-vast-frog"
    / "adaptive_polish_2025-06-11-02-58-27AM"
)